# Task 2: Exploratory Data Analysis (EDA) & Business Intelligence

**Objective:**
To uncover patterns, trends, and relationships within the dataset (`ApexPlanet_DataAnalytics_Dataset (1).xlsx`) and execute business intelligence queries.

---
## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')

# Load dataset
file_path = 'ApexPlanet_DataAnalytics_Dataset (1).xlsx'
df = pd.read_excel(file_path, sheet_name='Sales_Dataset')

print('Dataset Shape:', df.shape)
df.head()

## 2. Data Cleaning & Descriptive Statistics (Univariate Analysis)

Inspecting missing values and calculating key summary metrics for numerical and categorical variables.

In [ ]:
# Check missing values
print('--- Missing Values ---')
print(df.isnull().sum())

# Handle missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['City'] = df['City'].fillna('Unknown')

# Summary statistics
print('\n--- Numerical Summary Statistics ---')
num_cols = ['Age', 'Quantity', 'Unit_Price', 'Total_Sales']
num_stats = df[num_cols].describe().T
num_stats['median'] = df[num_cols].median()
display(num_stats[['count', 'mean', 'median', 'std', 'min', 'max']])

print('\n--- Categorical Summary Statistics ---')
cat_cols = ['Gender', 'City', 'Product', 'Category']
display(df[cat_cols].describe().T)

## 3. Exploratory Visualizations & Multivariate Analysis

Generating visualizations to analyze distributions, product performance, city demand, and feature correlations.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Total Sales Distribution
sns.histplot(df['Total_Sales'], kde=True, ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Distribution of Total Sales')

# 2. Total Revenue by Category
sns.barplot(data=df, x='Category', y='Total_Sales', estimator=sum, ax=axes[0,1], palette='Blues_d', errorbar=None)
axes[0,1].set_title('Total Revenue by Category')
axes[0,1].tick_params(axis='x', rotation=30)

# 3. Total Revenue by City
sns.barplot(data=df, x='City', y='Total_Sales', estimator=sum, ax=axes[1,0], palette='Greens_d', errorbar=None)
axes[1,0].set_title('Total Revenue by City')
axes[1,0].tick_params(axis='x', rotation=30)

# 4. Age vs Total Sales by Gender
sns.scatterplot(data=df, x='Age', y='Total_Sales', hue='Gender', alpha=0.7, ax=axes[1,1])
axes[1,1].set_title('Age vs. Total Sales by Gender')

plt.tight_layout()
plt.savefig('eda_summary_plots.png', dpi=300)
plt.show()

# Correlation Matrix Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df[['Age', 'Quantity', 'Unit_Price', 'Total_Sales']].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Matrix Heatmap')
plt.show()

## 4. SQL Business Questions Implementation

Replicating SQL aggregation logic to extract strategic business metrics.

In [ ]:
# Q1: Top 5 Products by Revenue
print('=== Q1: Top 5 Products by Revenue ===')
q1 = df.groupby('Product')['Total_Sales'].sum().reset_index().sort_values(by='Total_Sales', ascending=False).head(5)
display(q1)

# Q2: Monthly Sales Trend
print('\n=== Q2: Monthly Sales Trend ===')
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Sales_Month'] = df['Order_Date'].dt.to_period('M')
q2 = df.groupby('Sales_Month').agg(
    Total_Orders=('Order_ID', 'nunique'),
    Total_Revenue=('Total_Sales', 'sum')
).reset_index()
display(q2.head(6))

# Q3: Revenue by Category & Gender
print('\n=== Q3: Revenue Breakdown by Category & Gender ===')
q3 = df.groupby(['Category', 'Gender']).agg(
    Total_Orders=('Order_ID', 'count'),
    Total_Revenue=('Total_Sales', 'sum')
).reset_index().sort_values(by=['Category', 'Total_Revenue'], ascending=[True, False])
display(q3)

# Q4: Top Regional Markets by Revenue and AOV
print('\n=== Q4: Top Regional Cities by Revenue & AOV ===')
q4 = df[df['City'] != 'Unknown'].groupby('City').agg(
    Total_Orders=('Order_ID', 'count'),
    Total_Revenue=('Total_Sales', 'sum'),
    Average_Order_Value=('Total_Sales', 'mean')
).reset_index().sort_values(by='Total_Revenue', ascending=False).head(5)
display(q4)